# Open-Meteo Weather Dashboard
Fetches and visualises current conditions + 7-day forecast for any city.

In [ ]:
import requests
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from datetime import datetime

CITY = "Vienna"  # Change this to any city

In [ ]:
WMO = {
    0: "Clear sky", 1: "Mainly clear", 2: "Partly cloudy", 3: "Overcast",
    45: "Foggy", 48: "Icy fog",
    51: "Light drizzle", 53: "Moderate drizzle", 55: "Dense drizzle",
    61: "Slight rain", 63: "Moderate rain", 65: "Heavy rain",
    71: "Slight snow", 73: "Moderate snow", 75: "Heavy snow",
    80: "Slight showers", 81: "Moderate showers", 82: "Violent showers",
    95: "Thunderstorm", 96: "Thunderstorm w/ hail", 99: "Thunderstorm w/ heavy hail"
}

# --- Geocoding ---
geo = requests.get(
    "https://geocoding-api.open-meteo.com/v1/search",
    params={"name": CITY, "count": 1, "language": "en", "format": "json"}
).json()

if not geo.get("results"):
    raise ValueError(f"City '{CITY}' not found.")

loc = geo["results"][0]
lat, lon = loc["latitude"], loc["longitude"]
label = f"{loc['name']}, {loc['country']}"
print(f"Location: {label} ({lat:.3f}, {lon:.3f})")

# --- Weather data ---
data = requests.get(
    "https://api.open-meteo.com/v1/forecast",
    params={
        "latitude": lat,
        "longitude": lon,
        "current": "temperature_2m,relative_humidity_2m,apparent_temperature,wind_speed_10m,weathercode",
        "daily": "weathercode,temperature_2m_max,temperature_2m_min",
        "wind_speed_unit": "kmh",
        "timezone": "auto",
        "forecast_days": 7
    }
).json()

current = data["current"]
daily   = data["daily"]

dates    = [datetime.strptime(d, "%Y-%m-%d") for d in daily["time"]]
day_labels = []
for i, d in enumerate(dates):
    if i == 0: day_labels.append("Today")
    elif i == 1: day_labels.append("Tomorrow")
    else: day_labels.append(d.strftime("%a"))

t_max = daily["temperature_2m_max"]
t_min = daily["temperature_2m_min"]
codes = daily["weathercode"]

In [ ]:
plt.style.use("dark_background")
fig = plt.figure(figsize=(14, 9), facecolor="#0f172a")
fig.suptitle(f"Weather — {label}", fontsize=16, color="#94a3b8", y=0.97)

gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.45, wspace=0.35)

CARD  = "#1e293b"
BLUE  = "#3b82f6"
CYAN  = "#22d3ee"
MUTED = "#64748b"
TEXT  = "#e2e8f0"

# ── 1. Current conditions (text panel) ─────────────────────────────────────
ax0 = fig.add_subplot(gs[0, 0])
ax0.set_facecolor(CARD)
ax0.set_xlim(0, 1)
ax0.set_ylim(0, 1)
ax0.axis("off")
ax0.set_title("Current Conditions", color=MUTED, fontsize=10, pad=8)

ax0.text(0.5, 0.82, f"{round(current['temperature_2m'])}°C",
         ha="center", va="center", fontsize=42, fontweight="bold", color=TEXT, transform=ax0.transAxes)
ax0.text(0.5, 0.60, WMO.get(current["weathercode"], "Unknown"),
         ha="center", va="center", fontsize=12, color=CYAN, transform=ax0.transAxes)
ax0.text(0.5, 0.45, f"Feels like {round(current['apparent_temperature'])}°C",
         ha="center", va="center", fontsize=10, color=MUTED, transform=ax0.transAxes)

metrics = [
    ("Humidity", f"{current['relative_humidity_2m']}%"),
    ("Wind",     f"{round(current['wind_speed_10m'])} km/h"),
]
for i, (lbl, val) in enumerate(metrics):
    x = 0.25 + i * 0.5
    ax0.text(x, 0.24, lbl, ha="center", fontsize=8, color=MUTED, transform=ax0.transAxes)
    ax0.text(x, 0.10, val, ha="center", fontsize=14, fontweight="bold", color=TEXT, transform=ax0.transAxes)

# ── 2. 7-day temperature range ──────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, 1])
ax1.set_facecolor(CARD)
ax1.set_title("7-Day Temperature Range", color=MUTED, fontsize=10, pad=8)

x = range(len(day_labels))
ax1.fill_between(x, t_min, t_max, alpha=0.25, color=BLUE)
ax1.plot(x, t_max, "o-", color=BLUE,  linewidth=2, markersize=5, label="High")
ax1.plot(x, t_min, "o-", color=CYAN,  linewidth=2, markersize=5, label="Low",  linestyle="--")

for i, (hi, lo) in enumerate(zip(t_max, t_min)):
    ax1.text(i, hi + 0.4, f"{round(hi)}°", ha="center", fontsize=7.5, color=BLUE)
    ax1.text(i, lo - 0.8, f"{round(lo)}°", ha="center", fontsize=7.5, color=CYAN)

ax1.set_xticks(list(x))
ax1.set_xticklabels(day_labels, fontsize=8, color=TEXT)
ax1.tick_params(colors=MUTED)
ax1.set_ylabel("°C", color=MUTED, fontsize=9)
ax1.spines[:].set_color("#334155")
ax1.yaxis.label.set_color(MUTED)
ax1.tick_params(axis="y", colors=MUTED)
ax1.legend(fontsize=8, facecolor=CARD, edgecolor="#334155", labelcolor=TEXT)

# ── 3. Humidity gauge (horizontal bar) ─────────────────────────────────────
ax2 = fig.add_subplot(gs[1, 0])
ax2.set_facecolor(CARD)
ax2.set_title("Humidity & Wind", color=MUTED, fontsize=10, pad=8)
ax2.axis("off")

hum = current["relative_humidity_2m"]
wind = current["wind_speed_10m"]

def draw_bar(ax, y, value, vmax, color, label, unit):
    ax.barh(y, vmax, height=0.3, color="#334155", left=0)
    ax.barh(y, value, height=0.3, color=color, left=0)
    ax.text(-0.02, y, label, ha="right", va="center", fontsize=9, color=MUTED, transform=ax.get_yaxis_transform())
    ax.text(vmax + vmax * 0.02, y, f"{round(value)}{unit}", ha="left", va="center", fontsize=9,
            color=TEXT, transform=ax.transData)

ax2.set_xlim(0, 120)
ax2.set_ylim(0, 1)
draw_bar(ax2, 0.7, hum,  100, CYAN,  "Humidity", "%")
draw_bar(ax2, 0.3, min(wind, 100), 100, BLUE, "Wind",     " km/h")

# ── 4. Conditions table ─────────────────────────────────────────────────────
ax3 = fig.add_subplot(gs[1, 1])
ax3.set_facecolor(CARD)
ax3.set_title("7-Day Conditions", color=MUTED, fontsize=10, pad=8)
ax3.axis("off")

col_labels = ["Day", "Condition", "High", "Low"]
rows = [
    [day_labels[i], WMO.get(codes[i], "—"), f"{round(t_max[i])}°C", f"{round(t_min[i])}°C"]
    for i in range(7)
]

table = ax3.table(
    cellText=rows,
    colLabels=col_labels,
    loc="center",
    cellLoc="center"
)
table.auto_set_font_size(False)
table.set_fontsize(8.5)
table.scale(1, 1.35)

for (row, col), cell in table.get_celld().items():
    cell.set_facecolor(CARD if row > 0 else "#0f172a")
    cell.set_text_props(color=TEXT if row > 0 else MUTED)
    cell.set_edgecolor("#334155")
    if col == 2 and row > 0: cell.get_text().set_color(BLUE)
    if col == 3 and row > 0: cell.get_text().set_color(CYAN)

plt.savefig("weather_dashboard.png", dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
plt.show()
print("Saved to weather_dashboard.png")